# Favorite World — Train Your Own Cartesia-Quality Hindi & English AI Voice (Free Google Colab T4)

**Zero PC CPU/GPU load.** Trains an **XTTS-v2** multilingual voice model (Hindi + English + Hinglish) on **all your Favorite World voiceovers** using Google's free cloud T4 GPU, saves the trained model permanently in your **Google Drive**, and gives you a live `gradio.live` link that plugs straight into **Voiceover Studio Pro**.

### Quick Start:
1. Make sure **Runtime → Change runtime type → T4 GPU** is selected.
2. Upload `out/favorite_world_voice_dataset.zip` to Google Drive at `MyDrive/FavoriteWorld_Voice_Model/favorite_world_voice_dataset.zip` (or upload when Cell 2 asks).
3. Run **Cells 1 → 2 → 3** once to train (~20–30 min) and save your voice to Drive.
4. Next time you want voiceovers, run **Cell 1** and **Cell 4 (Serve Saved Model)** — loads in ~20 seconds!

In [ ]:
# CELL 1: Mount Google Drive & Install Cloud TTS Training Stack
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi -L
!pip install -q "numpy<2.0.0" "coqui-tts>=0.25.0" "gradio==5.23.1" soundfile


In [ ]:
# CELL 2: Load Your Favorite World Voiceover Dataset
import os, zipfile

DRIVE_DIR = "/content/drive/MyDrive/FavoriteWorld_Voice_Model"
os.makedirs(DRIVE_DIR, exist_ok=True)
DRIVE_ZIP = os.path.join(DRIVE_DIR, "favorite_world_voice_dataset.zip")
DATASET_DIR = "/content/fw_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

if not os.path.exists(DRIVE_ZIP):
    from google.colab import files
    print("Upload favorite_world_voice_dataset.zip (from your clausd/out/ folder):")
    uploaded = files.upload()
    up_name = next(iter(uploaded.keys()))
    os.replace(up_name, DRIVE_ZIP)

with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(DATASET_DIR)

!cat /content/fw_dataset/dataset_info.json


In [ ]:
# CELL 3: Fine-Tune XTTS-v2 on Your Voiceovers (Runs on Google Cloud T4 GPU)
import os, shutil, glob
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig
from TTS.utils.manage import ModelManager

OUT_PATH = "/content/xtts_runs"
SAVED_MODEL_DIR = "/content/drive/MyDrive/FavoriteWorld_Voice_Model/trained_xtts"
DATASET_DIR = "/content/fw_dataset"
os.makedirs(OUT_PATH, exist_ok=True)
os.makedirs(SAVED_MODEL_DIR, exist_ok=True)

# Download base XTTS-v2 checkpoints (accept Coqui CPML license automatically in Colab)
os.environ["COQUI_TOS_AGREED"] = "1"
MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
mm = ModelManager()
model_path, _, _ = mm.download_model(MODEL_NAME)
if not os.path.isdir(model_path):
    model_path = os.path.dirname(model_path)

DVAE_CHECKPOINT = os.path.join(model_path, "dvae.pth")
MEL_NORM_FILE = os.path.join(model_path, "mel_stats.pth")
XTTS_CHECKPOINT = os.path.join(model_path, "model.pth")
TOKENIZER_FILE = os.path.join(model_path, "vocab.json")
CONFIG_FILE = os.path.join(model_path, "config.json")

config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="favorite_world",
    path=DATASET_DIR,
    meta_file_train=os.path.join(DATASET_DIR, "metadata.csv"),
    language="hi",
)

audio_config = XttsAudioConfig(sample_rate=24000, dvae_checkpoint=DVAE_CHECKPOINT, mel_norm_file=MEL_NORM_FILE)
model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=288000,
    max_text_length=320,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name="fw_xtts_hi_en",
    project_name="FavoriteWorldVoice",
    dashboard_logger="tensorboard",
    audio=audio_config,
    batch_size=2,
    batch_group_size=16,
    eval_batch_size=1,
    num_loader_workers=2,
    eval_split_max_size=64,
    print_step=25,
    plot_step=200,
    save_step=500,
    save_n_checkpoints=1,
    save_best_after=250,
    epochs=10,
    optimizer="AdamW",
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000, 150000, 300000], "gamma": 0.5, "last_epoch": -1},
    grad_clip=1.0,
    test_sentences=[],
    datasets=[config_dataset],
)

train_samples, eval_samples = load_tts_samples(
    [config_dataset],
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=0.05,
)

model = GPTTrainer.init_from_config(config)
trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False, start_with_eval=False),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

# Copy best checkpoint + config + vocab + speaker_ref.wav to Google Drive
best_ckpts = sorted(glob.glob(f"{OUT_PATH}/**/best_model*.pth", recursive=True), key=os.path.getmtime)
latest_ckpts = sorted(glob.glob(f"{OUT_PATH}/**/checkpoint_*.pth", recursive=True), key=os.path.getmtime)
chosen = best_ckpts[-1] if best_ckpts else latest_ckpts[-1]
shutil.copy2(chosen, os.path.join(SAVED_MODEL_DIR, "model.pth"))
shutil.copy2(CONFIG_FILE, os.path.join(SAVED_MODEL_DIR, "config.json"))
shutil.copy2(TOKENIZER_FILE, os.path.join(SAVED_MODEL_DIR, "vocab.json"))
shutil.copy2(os.path.join(DATASET_DIR, "speaker_ref.wav"), os.path.join(SAVED_MODEL_DIR, "speaker_ref.wav"))
print("SAVED TRAINED VOICE MODEL TO GOOGLE DRIVE:", SAVED_MODEL_DIR)


In [ ]:
# CELL 4: Serve Your Trained Favorite World Voice (Works with Voiceover Studio Pro!)
# Run this cell anytime after training — loads directly from Google Drive in ~20 seconds.
import os, tempfile, torch, torchaudio, gradio as gr
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

SAVED_MODEL_DIR = "/content/drive/MyDrive/FavoriteWorld_Voice_Model/trained_xtts"
DEFAULT_REF = os.path.join(SAVED_MODEL_DIR, "speaker_ref.wav")

cfg = XttsConfig()
cfg.load_json(os.path.join(SAVED_MODEL_DIR, "config.json"))
xtts = Xtts.init_from_config(cfg)
xtts.load_checkpoint(cfg, checkpoint_dir=SAVED_MODEL_DIR, use_deepspeed=False)
xtts.cuda()
print("Loaded Favorite World Trained Voice!")

def generate_tts_audio(text_input, language_id, audio_prompt_path_input, exaggeration_input, temperature_input, seed_num_input, cfgw_input):
    ref = audio_prompt_path_input if audio_prompt_path_input else DEFAULT_REF
    lang = "hi" if str(language_id).lower().startswith("hi") else "en"
    if int(seed_num_input or 0) > 0:
        torch.manual_seed(int(seed_num_input))
    gpt_cond_latent, speaker_embedding = xtts.get_conditioning_latents(audio_path=[ref])
    out = xtts.inference(
        text=text_input,
        language=lang,
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=float(temperature_input or 0.7),
        repetition_penalty=2.5,
        top_k=50,
        top_p=0.85,
        speed=max(0.75, min(1.3, 0.8 + float(cfgw_input or 0.5) * 0.4)),
    )
    out_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    torchaudio.save(out_wav, torch.tensor(out["wav"]).unsqueeze(0), 24000)
    return out_wav

with gr.Blocks() as demo:
    gr.Markdown("# Favorite World — Trained Cloud Voice Server (Hindi & English)")
    txt = gr.Textbox(label="Text")
    lang = gr.Dropdown(["hi", "en"], value="hi", label="Language")
    ref = gr.Audio(type="filepath", label="Optional Override Reference Audio")
    ex = gr.Slider(0.25, 1.2, value=0.55, label="Exaggeration")
    temp = gr.Slider(0.2, 1.2, value=0.7, label="Temperature")
    seed = gr.Number(value=0, label="Seed")
    cfgw = gr.Slider(0.2, 1.0, value=0.5, label="Speed/Pace")
    btn = gr.Button("Generate")
    out = gr.Audio(type="filepath", label="Output")
    btn.click(generate_tts_audio, [txt, lang, ref, ex, temp, seed, cfgw], [out], api_name="generate_tts_audio")

demo.queue().launch(share=True, debug=True)
